In [1]:
from transformers import AutoProcessor, AutoModel
import torch, os
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# define a fixed local cache path
cache_dir = os.path.join(os.getcwd(), "..","models", "siglip")
os.makedirs(cache_dir, exist_ok=True)

model_id = "google/siglip-base-patch16-224"
processor = AutoProcessor.from_pretrained(model_id, cache_dir=cache_dir)
model = AutoModel.from_pretrained(model_id, cache_dir=cache_dir).to(device)



Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
# --- test image and labels ---
image = Image.open("../dog.jpg")

labels = [ "animal", "dog", "chair","dick","cat",]

inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits_per_image   # [1, len(labels)]
    probs = logits.softmax(dim=1)

top_idx = probs.argmax().item()
print(f"\nTop prediction: {labels[top_idx]} ({probs[0, top_idx]:.3f})")


Top prediction: dog (0.742)


In [7]:
from transformers import AutoProcessor, OmDetTurboForObjectDetection
import torch
from PIL import Image

model_id = "omlab/omdet-turbo-swin-tiny-hf"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = OmDetTurboForObjectDetection.from_pretrained(model_id).to(device)

image = Image.open("../env.jpg")
text_labels = ["any", "object", "grass", "flower", "car"]  # you can supply many labels

inputs = processor(image, text=text_labels, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_grounded_object_detection(
    outputs=outputs,
    threshold=0.02,                # confidence threshold
    target_sizes=[image.size[::-1]]
)[0]

for scores, labels, boxs in zip(results["scores"], results["labels"], results["boxes"]):
    print(f"Detected {labels} with confidence {scores:.2f} at {boxs.tolist()}")


Detected 3 with confidence 0.48 at [0.5245490074157715, 41.1217155456543, 288.03863525390625, 259.6097717285156]
Detected 3 with confidence 0.25 at [103.10107421875, 164.71327209472656, 189.65518188476562, 229.38180541992188]
Detected 1 with confidence 0.23 at [1.003218173980713, 0.0, 463.0, 94.8740005493164]
Detected 2 with confidence 0.23 at [0.5245490074157715, 41.1217155456543, 288.03863525390625, 259.6097717285156]
Detected 3 with confidence 0.20 at [235.6027069091797, 223.02256774902344, 261.6225280761719, 255.0729522705078]
Detected 3 with confidence 0.20 at [127.96858978271484, 119.34400939941406, 175.5240478515625, 151.39694213867188]
Detected 4 with confidence 0.17 at [349.4828796386719, 86.0514907836914, 368.89532470703125, 97.02552795410156]
Detected 3 with confidence 0.17 at [183.62567138671875, 232.57176208496094, 201.7171173095703, 244.85926818847656]
Detected 3 with confidence 0.16 at [204.75613403320312, 166.5525665283203, 232.1109161376953, 187.38450622558594]
Detecte

In [4]:
results

{'boxes': tensor([[  0.5245,  41.1217, 288.0386, 259.6098]], device='cuda:0'),
 'scores': tensor([0.4824], device='cuda:0'),
 'labels': tensor([3], device='cuda:0'),
 'text_labels': None}

In [1]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torch
from PIL import Image

image = Image.open("../env.jpg")
text = "all objects"  # can also be: "cup, table, book, person"

processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny")

inputs = processor(images=image, text=text, return_tensors="pt")
outputs = model(**inputs)
results = processor.post_process_grounded_object_detection(outputs, inputs, box_threshold=0.3, text_threshold=0.25)

for box, label in zip(results[0]["boxes"], results[0]["labels"]):
    print(label, box.tolist())


: 

In [ ]:
import torch.optim.adam
import torch.optim.sgd